# Combined Workflow — Postcodes, Geocoding & Routing

An end-to-end scenario tying all four services together: given two UK postcodes,
look up their metadata, geocode both locations, calculate driving and walking routes
between them, and render everything on a single map.

## Prerequisites

| Service        | Default port | Role                            |
| -------------- | ------------ | ------------------------------- |
| Nominatim      | 8080         | Forward geocoding               |
| Photon         | 2322         | Reverse geocoding / enrichment  |
| OSRM + HAProxy | 80           | Road routing (driving, walking) |
| postcodes.io   | 8000         | UK postcode / outcode metadata  |


______________________________________________________________________

## Start Services

Before running this notebook, make sure the required Docker services are up.
From the **project root**, run:

```bash
make up                  # starts all services
```

Check running containers with `make ps`. See the main
[README](../README.md#docker-services) for all available profiles and
configuration details.


## Setup — Imports and Configuration


In [ ]:
import importlib

import folium
import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder, lookup_postcode, route
from airgap_geo.settings import (
    NOMINATIM_URL,
    OSRM_API,
    PHOTON_API,
    POSTCODES_URL,
)

client = httpx.AsyncClient()

print("Configured service endpoints")
print(f"  Nominatim    : {NOMINATIM_URL}")
print(f"  Photon       : {PHOTON_API}")
print(f"  OSRM/HAProxy : {OSRM_API}")
print(f"  postcodes.io : {POSTCODES_URL}")


def _decode_polyline(encoded: str) -> list[tuple[float, float]]:
    """Decode an OSRM / Google encoded polyline string into (lat, lon) pairs."""
    coords: list[tuple[float, float]] = []
    index = 0
    lat = 0
    lng = 0
    while index < len(encoded):
        for is_lng in (False, True):
            shift, result = 0, 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1F) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = ~(result >> 1) if (result & 1) else (result >> 1)
            if is_lng:
                lng += delta
            else:
                lat += delta
        coords.append((lat / 1e5, lng / 1e5))
    return coords

## Health Check


In [ ]:
rows = []
for name, base_url in _ENDPOINTS.items():
    try:
        r = requests.get(base_url, timeout=5)
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": r.status_code,
                "Reachable": "\u2713",
            }
        )
    except Exception:
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": "ERR",
                "Reachable": "\u2717",
            }
        )

pd.DataFrame(rows).set_index("Service")

______________________________________________________________________

## End-to-End: Postcodes \\u2192 Geocode \\u2192 Route \\u2192 Map

Given two UK postcodes, geocode both locations, calculate driving and walking
routes between them, and render both on a single map with the postcode metadata
shown in the marker popups.


In [ ]:
# --- Inputs -----------------------------------------------------------
POSTCODE_A = "EC2V 6DN"  # Bank of England
POSTCODE_B = "WC2N 5DU"  # Trafalgar Square

# --- Step 1: postcode metadata ----------------------------------------
pc_a = await lookup_postcode(POSTCODE_A, client)
pc_b = await lookup_postcode(POSTCODE_B, client)

# --- Step 2: geocode --------------------------------------------------
geo_a = await geocoder(POSTCODE_A, client)
geo_b = await geocoder(POSTCODE_B, client)

coords_a = (geo_a.geo.lat, geo_a.geo.lon) if geo_a else None
coords_b = (geo_b.geo.lat, geo_b.geo.lon) if geo_b else None

print(f"{POSTCODE_A}  \u2192  {coords_a}  ({pc_a.admin_district if pc_a else '?'})")
print(f"{POSTCODE_B}  \u2192  {coords_b}  ({pc_b.admin_district if pc_b else '?'})")

# --- Step 3: routing --------------------------------------------------
route_driving = await route(coords_a, coords_b, client, profile="driving")
route_walking = await route(coords_a, coords_b, client, profile="walking")


def _route_summary(r, label: str) -> None:
    if r and r.routes:
        rs = r.routes[0]
        print(
            f"{label:10s}  distance={rs.distance_m / 1000:.2f} km  duration={rs.duration_s / 60:.1f} min"
        )
    else:
        print(f"{label:10s}  no route returned")


_route_summary(route_driving, "driving")
_route_summary(route_walking, "walking")

In [ ]:
# --- Step 4: combined map ---------------------------------------------
mid_lat = (coords_a[0] + coords_b[0]) / 2
mid_lon = (coords_a[1] + coords_b[1]) / 2
m_combined = folium.Map(location=[mid_lat, mid_lon], zoom_start=14)

for coords, pc_data, label, colour in [
    (coords_a, pc_a, POSTCODE_A, "blue"),
    (coords_b, pc_b, POSTCODE_B, "red"),
]:
    popup_html = (
        f"<b>{label}</b><br>"
        f"District: {pc_data.admin_district or '-' if pc_data else '-'}<br>"
        f"Ward: {pc_data.admin_ward or '-' if pc_data else '-'}<br>"
        f"Region: {pc_data.region or '-' if pc_data else '-'}"
    )
    folium.Marker(
        list(coords),
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=label,
        icon=folium.Icon(color=colour, icon="info-sign"),
    ).add_to(m_combined)

_ROUTE_STYLES = {
    "driving": (route_driving, "royalblue", "Driving route"),
    "walking": (route_walking, "darkorange", "Walking route"),
}
for _profile, (r_data, colour, tip) in _ROUTE_STYLES.items():
    if r_data and r_data.routes:
        encoded = r_data.routes[0].geometry or ""
        if encoded:
            folium.PolyLine(
                _decode_polyline(encoded),
                color=colour,
                weight=4,
                opacity=0.8,
                tooltip=tip,
            ).add_to(m_combined)

m_combined

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()